# Prototype: Convert nuScenes JSON to KITTI `.txt` Predictions

This notebook prototypes reading a standard `results.json` generated by a nuScenes evaluation run, mapping the sample tokens to KITTI frame indices, applying the correct coordinate transformations, and exporting `.txt` files that the KITTI evaluator can read.

In [15]:
import os
import json
import pickle
import numpy as np
from pathlib import Path
from pyquaternion import Quaternion
from nuscenes.nuscenes import NuScenes
from nuscenes.utils.data_classes import Box

## 1. Setup Paths and Load Data
Replace `RESULTS_JSON` with the path to your actual nuScenes predictions.

In [16]:
RESULTS_JSON = '/OpenPCDet/output/OpenPCDet/tools/cfgs/kitti_models/second_nuscenes/default/eval/epoch_7862/test/default/final_result/data/results_nusc.json'
# Use the native OpenPCDet nuScenes infos instead of the converted KITTI DB!
NUSC_INFOS = '/OpenPCDet/datasets/nuscenes/v1.0-mini/nuscenes_infos_10sweeps_val.pkl'  # UPDATE THIS
NUSC_ROOT = '/OpenPCDet/datasets/nuscenes/v1.0-mini'
OUTPUT_DIR = Path('/OpenPCDet/tools/kitti_style_predictions')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [17]:
# Load NuScenes dataset context (needed for calibration matrices)
nusc = NuScenes(version='v1.0-mini', dataroot=NUSC_ROOT, verbose=True)

Loading NuScenes tables for version v1.0-mini...
23 category,
8 attribute,
4 visibility,
911 instance,
12 sensor,
120 calibrated_sensor,
31206 ego_pose,
8 log,
10 scene,
404 sample,
31206 sample_data,
18538 sample_annotation,
4 map,
Done loading in 0.3 seconds.
Reverse indexing ...
Done reverse indexing in 0.1 seconds.


## 2. Create Token -> KITTI Matcher
OpenPCDet uses `kitti_infos_val.pkl` to read the converted dataset. We need this to match the `sample_token` from the nuScenes JSON back to the `000000`-style KITTI IDs (often just 0 to N-1 based on the lists).

In [18]:
def get_token_mapping(infos_path):
    """
    Reads the mapping directly from the native OpenPCDet nuScenes infos.
    The order in this list perfectly matches the dataloader indices yielding 000000.txt, 000001.txt, etc.
    """
    with open(infos_path, 'rb') as f:
        infos = pickle.load(f)
    
    token_to_idx = {}
    for idx, info in enumerate(infos):
        # info['token'] is the 32-character nuScenes sample token
        token_to_idx[info['token']] = idx
        
    return token_to_idx

token_to_idx = get_token_mapping(NUSC_INFOS)
print(f"Loaded {len(token_to_idx)} mapping entries directly from nuScenes infos.")

Loaded 81 mapping entries directly from nuScenes infos.


## 3. Coordinate Transformation Mapping
The predictions in the JSON are in the **nuScenes global coordinate frame**. We need to map them back to the **KITTI camera_0 frame**.

In [19]:
def convert_box_to_kitti_str(box: Box, kitti_class_name: str, score: float):
    """
    Transforms a nuscenes Box (already projected into KITTI cam coordinate frame)
    into the KITTI format text line.
    """
    # Extract w, l, h from nuScenes box (which correspond to w, l, h, but KITTI stores h, w, l)
    w, l, h = box.wlh
    x, y, z = box.center
    
    # Extraction of rotation around Y axis (KITTI ry)
    v = np.dot(box.orientation.rotation_matrix, np.array([1, 0, 0]))
    yaw = np.arctan2(v[2], v[0])
    
    # Default placeholders for KITTI 2D bboxes when evaluating 3D: [0, 0, 50, 50] 
    # Default placeholders for truncation (0) and occlusion (0), alpha (-10)
    kitti_str = f"{kitti_class_name} 0 0 -10 0.0 0.0 50.0 50.0 {h:.4f} {w:.4f} {l:.4f} {x:.4f} {y:.4f} {z:.4f} {yaw:.4f} {score:.4f}\n"
    return kitti_str

## 4. Iterate over `results_nusc.json`
Assuming we generated the maps and functions successfully, we execute the looping write.

In [20]:
def process_results(results_path, nusc, token_to_idx, output_dir):
    with open(results_path, 'r') as f:
        data = json.load(f)
        
    results_dict = data.get('results', data)
    
    # KITTI uses camera_0 (front camera)
    # The nuscenes to KITTI camera transform
    kitti_to_nu_lidar = Quaternion(axis=(0, 0, 1), angle=np.pi / 2)
    kitti_to_nu_lidar_inv = kitti_to_nu_lidar.inverse
    
    empty_frames = 0
    count = 0
    
    # Iterate over the GT mapping to ensure exact 1:1 file creation
    for sample_token, frame_idx in token_to_idx.items():
        txt_path = output_dir / f"{frame_idx:06d}.txt"
        
        # 1. Get sample and sensors
        sample = nusc.get('sample', sample_token)
        cam_front_data = nusc.get('sample_data', sample['data']['CAM_FRONT'])
        cs_record = nusc.get('calibrated_sensor', cam_front_data['calibrated_sensor_token'])
        pose_record = nusc.get('ego_pose', cam_front_data['ego_pose_token'])
        
        lines = []
        predictions = results_dict.get(sample_token, [])
        if not predictions:
            empty_frames += 1
            
        for pred in predictions:
            # Reconstruct the Box in global coordinates
            box = Box(
                pred['translation'], 
                pred['size'], 
                Quaternion(pred['rotation']),
                name=pred['detection_name'], 
                score=pred['detection_score']
            )
            
            # Map classes (example: car -> Car)
            kitti_class = box.name.capitalize()
            if box.name == 'bicycle': kitti_class = 'Cyclist'
            
            # Move box to ego vehicle coordinate system
            box.translate(-np.array(pose_record['translation']))
            box.rotate(Quaternion(pose_record['rotation']).inverse)

            # Move box to sensor coordinate system (camera_0)
            box.translate(-np.array(cs_record['translation']))
            box.rotate(Quaternion(cs_record['rotation']).inverse)
            
            w, l, h = box.wlh
            x, y, z = box.center
            
            v = np.dot(box.orientation.rotation_matrix, np.array([1, 0, 0]))
            yaw = np.arctan2(v[2], v[0])
            
            lines.append(f"{kitti_class} 0 0 -10 0.0 0.0 50.0 50.0 {h:.4f} {w:.4f} {l:.4f} {x:.4f} {y:.4f} {z:.4f} {yaw:.4f} {box.score:.4f}\n")
            
        with open(txt_path, 'w') as f:
            f.writelines(lines)
            
        count += 1
        
    print(f"Exported {count} prediction frames. Empty frames: {empty_frames}")

# Run the processor 
process_results(RESULTS_JSON, nusc, token_to_idx, OUTPUT_DIR)

Exported 81 prediction frames. Empty frames: 0


## 5. Calculate KITTI Evaluation Metrics
To calculate the metrics locally, we need the Ground Truth labels in the exact same `.txt` format. Since you already ran the `export_kitti.py` converter earlier, the easiest and most accurate way is to use the `label_2` directory it generated (it handles tricky KITTI definitions like occlusion and truncation levels for easy/mod/hard splits).

We can load both your written prediction `.txt` directory and the Ground Truth `.txt` directory using OpenPCDet's native KITTI modules, and then run the evaluator!

In [21]:
from pcdet.datasets.kitti.kitti_object_eval_python import kitti_common
from pcdet.datasets.kitti.kitti_object_eval_python.eval import get_official_eval_result

# To get accurate Easy/Moderate/Hard splits, we MUST use the Ground Truth generated by `export_kitti.py`.
# nuScenes does not natively have occlusion/truncation. The ONLY way `export_kitti.py` splits difficulties
# is by projecting each 3D box into the 2D image plane and categorizing by pixel height (Easy >= 40px, Mod >= 25px).
# Using the pre-converted label_2 directory gives us those accurate 2D bounding box attributes.

# UPDATE: Pointing to the v1.0-mini converted split to match your current subset
GT_DIR = '/OpenPCDet/datasets_converted/nuscenes/v1.0-mini_kitti_format/mini_val/label_2'
PRED_DIR = str(OUTPUT_DIR)
LOG_FILE = Path('/OpenPCDet/tools/kitti_eval_results_log.txt')

# 1. Load lists of dicts required by the evaluator
print("Loading Ground Truth and Predictions...")
gt_annos = kitti_common.get_label_annos(GT_DIR)
dt_annos = kitti_common.get_label_annos(PRED_DIR)

# Align the lengths (in case some frames matched missing tokens)
if len(gt_annos) != len(dt_annos):
    print(f"Warning: Lengths don't match. GT has {len(gt_annos)}, PRED has {len(dt_annos)}.")

# 2. Evaluate 'Car' (Class 0 in KITTI mapping)
print("\n--- Car Evaluation ---")
car_result = get_official_eval_result(gt_annos, dt_annos, 0)
print(car_result[0] if isinstance(car_result, tuple) else car_result)

# Write to log file
with open(LOG_FILE, 'w') as f:
    f.write("--- Car Evaluation ---\n")
    text_to_write = car_result[0] if isinstance(car_result, tuple) else str(car_result)
    f.write(text_to_write + "\n")
    
    # Evaluate 'Pedestrian' (Class 1) and 'Cyclist' / Bicycle (Class 2) 
    ped_result = get_official_eval_result(gt_annos, dt_annos, 1)
    f.write("\n--- Pedestrian ---\n")
    f.write(ped_result[0] if isinstance(ped_result, tuple) else str(ped_result) + "\n")

    cyc_result = get_official_eval_result(gt_annos, dt_annos, 2)
    f.write("\n--- Cyclist ---\n")
    f.write(cyc_result[0] if isinstance(cyc_result, tuple) else str(cyc_result) + "\n")

print(f"\nResults successfully saved to {LOG_FILE}")

Loading Ground Truth and Predictions...

--- Car Evaluation ---


AssertionError: 